# 06 — Vision + Classification v2

Comprehensive vision analysis + two-stage classification on 50 randomly selected TikToks.
Includes subtitles, comments, and full metadata in all prompts.

**Cost estimate**: ~$0.70 total

**Cells**:
0. Setup
1. Sample Selection (50 from recent 500)
2. Apify Fetch
3. Download Media + Fetch Comments
4. Stage 1 — Comprehensive Perception
5. Stage 2 — 5-Facet Classification (vision-informed)
6. Stage 2 — 5-Facet Classification (text-only baseline)
7. 8-Facet Classification (redundancy test)
8. Slideshow Image Informativeness
9. Results Summary + Stratified Analysis
10. Generate Eval UI

In [ ]:
import sys, json, os, re, time, random, base64, subprocess
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field

import httpx
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# ── Paths ───────────────────────────────────────────────────────────────
# Find repo root by looking for CLAUDE.md
_cwd = Path.cwd()
REPO_ROOT = _cwd
for parent in [_cwd] + list(_cwd.parents):
    if (parent / "CLAUDE.md").exists():
        REPO_ROOT = parent
        break

sys.path.insert(0, str(REPO_ROOT / "src" / "backend"))

load_dotenv(REPO_ROOT / "workbench" / ".env")
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
APIFY_API_TOKEN = os.environ["APIFY_API_TOKEN"]

DATA_DIR = REPO_ROOT / "workbench" / "data" / "vision_v2"
MEDIA_DIR = DATA_DIR / "media"
RESULTS_DIR = DATA_DIR / "results"
MANIFEST_PATH = DATA_DIR / "manifest.json"
for d in [DATA_DIR, MEDIA_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EXPORT_PATH = REPO_ROOT / "workbench" / "data" / "my-export" / "full_anonymized.json"

# ── Gemini Config ───────────────────────────────────────────────────────
GEMINI_API_BASE = "https://generativelanguage.googleapis.com/v1beta"
GEMINI_MODEL = "gemini-3-flash-preview"
INPUT_COST_PER_M = 0.50   # Gemini 3 Flash
OUTPUT_COST_PER_M = 3.00   # Gemini 3 Flash
MAX_OUTPUT_TOKENS = 8192

# ── Apify Config ────────────────────────────────────────────────────────
APIFY_ACTOR_ID = "clockworks~tiktok-scraper"
APIFY_POLL_S = 5
APIFY_MAX_WAIT_S = 600

_TIKTOK_VIDEO_ID_RE = re.compile(r"/video/(\d+)")

# ── Helper: safe JSON parse ─────────────────────────────────────────────
def safe_json_parse(text: str) -> dict:
    """Parse JSON from Gemini output, handling markdown fences and truncation."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text[3:]
        if "```" in text:
            text = text[:text.rfind("```")]
        text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Try to fix truncated JSON by closing open braces/brackets
        for closer in ["}", "]}", "]}}", "]}}]"]:
            try:
                return json.loads(text + closer)
            except json.JSONDecodeError:
                continue
        return {"_parse_error": True, "_raw_text": text[:5000]}

# ── Helper: VTT subtitle parsing ────────────────────────────────────────
def parse_vtt(vtt_text: str) -> str:
    """Strip VTT headers and timestamps, return plain transcript."""
    lines = []
    for line in vtt_text.split("\n"):
        line = line.strip()
        if not line or line.startswith("WEBVTT") or line.startswith("NOTE"):
            continue
        if "-->" in line:
            continue
        if line.isdigit():
            continue
        lines.append(line)
    # Deduplicate consecutive identical lines (VTT often repeats)
    deduped = []
    for line in lines:
        if not deduped or line != deduped[-1]:
            deduped.append(line)
    return " ".join(deduped)

# ── Helper: text richness tier ──────────────────────────────────────────
def get_text_tier(caption: str, has_subtitles: bool) -> str:
    clean = (caption or "").strip()
    if len(clean) >= 50 and has_subtitles:
        return "rich_text"
    elif len(clean) >= 50:
        return "caption_only"
    else:
        return "low_text"

# ── Helper: cost estimate ───────────────────────────────────────────────
def estimate_cost(input_tok: int, output_tok: int) -> float:
    return (input_tok * INPUT_COST_PER_M + output_tok * OUTPUT_COST_PER_M) / 1_000_000

# ── Helper: download file ───────────────────────────────────────────────
def download_file(url: str, dest: Path) -> Path | None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    try:
        resp = httpx.get(url, timeout=120, follow_redirects=True)
        resp.raise_for_status()
        dest.write_bytes(resp.content)
        return dest
    except Exception as e:
        print(f"  WARN: download failed: {str(e)[:80]}")
        return None

# ── Gemini Client ───────────────────────────────────────────────────────
def gemini_generate(parts: list[dict], max_tokens: int = MAX_OUTPUT_TOKENS,
                    temperature: float = 0.2, json_mode: bool = True,
                    tools: list[dict] | None = None) -> dict:
    """Call Gemini generateContent. Returns full API response dict."""
    gen_config = {"maxOutputTokens": max_tokens, "temperature": temperature}
    if json_mode:
        gen_config["responseMimeType"] = "application/json"
    body = {"contents": [{"parts": parts}], "generationConfig": gen_config}
    if tools:
        body["tools"] = tools

    resp = httpx.post(
        f"{GEMINI_API_BASE}/models/{GEMINI_MODEL}:generateContent",
        params={"key": GEMINI_API_KEY},
        json=body, timeout=120,
    )
    resp.raise_for_status()
    return resp.json()

def gemini_upload_file(file_path: str, mime_type: str) -> str:
    """Upload file to Gemini File API. Returns file_name (files/xxx)."""
    file_size = os.path.getsize(file_path)
    client = httpx.Client(timeout=300)
    resp = client.post(
        "https://generativelanguage.googleapis.com/upload/v1beta/files",
        params={"key": GEMINI_API_KEY},
        headers={
            "X-Goog-Upload-Protocol": "resumable",
            "X-Goog-Upload-Command": "start",
            "X-Goog-Upload-Header-Content-Length": str(file_size),
            "X-Goog-Upload-Header-Content-Type": mime_type,
        },
        json={"file": {"displayName": Path(file_path).name}},
    )
    resp.raise_for_status()
    upload_url = resp.headers["X-Goog-Upload-URL"]
    with open(file_path, "rb") as f:
        resp = client.put(upload_url, headers={
            "X-Goog-Upload-Offset": "0",
            "X-Goog-Upload-Command": "upload, finalize",
            "Content-Length": str(file_size),
        }, content=f.read())
    resp.raise_for_status()
    client.close()
    return resp.json()["file"]["name"]

def gemini_wait_for_file(file_name: str, timeout: int = 120) -> str:
    """Poll until Gemini file is ACTIVE. Returns file URI."""
    client = httpx.Client(timeout=30)
    elapsed = 0
    while elapsed < timeout:
        resp = client.get(f"{GEMINI_API_BASE}/{file_name}", params={"key": GEMINI_API_KEY})
        resp.raise_for_status()
        data = resp.json()
        if data.get("state") == "ACTIVE":
            client.close()
            return data.get("uri", f"https://generativelanguage.googleapis.com/v1beta/{file_name}")
        if data.get("state") == "FAILED":
            client.close()
            raise RuntimeError(f"File processing failed: {file_name}")
        time.sleep(2)
        elapsed += 2
    client.close()
    raise TimeoutError(f"File {file_name} not ACTIVE within {timeout}s")

def gemini_delete_file(file_name: str):
    try:
        httpx.delete(f"{GEMINI_API_BASE}/{file_name}", params={"key": GEMINI_API_KEY}, timeout=10)
    except Exception:
        pass

def image_to_inline(path: Path) -> dict:
    b64 = base64.b64encode(path.read_bytes()).decode()
    return {"inlineData": {"mimeType": "image/jpeg", "data": b64}}

# ── Apify Client ────────────────────────────────────────────────────────
def apify_fetch(platform_ids: list[str]) -> list[dict]:
    """Fetch items from Apify with full download options."""
    urls = [f"https://www.tiktok.com/share/video/{pid}/" for pid in platform_ids]
    headers = {"Authorization": f"Bearer {APIFY_API_TOKEN}", "Content-Type": "application/json"}
    client = httpx.Client(timeout=60)
    resp = client.post(f"https://api.apify.com/v2/acts/{APIFY_ACTOR_ID}/runs", headers=headers,
        json={"postURLs": urls, "resultsPerPage": len(urls),
              "shouldDownloadVideos": True, "shouldDownloadCovers": True,
              "shouldDownloadSlideshowImages": True, "shouldDownloadSubtitles": True,
              "downloadSubtitlesOptions": "DOWNLOAD_SUBTITLES", "commentsPerPost": 10})
    resp.raise_for_status()
    run_id = resp.json()["data"]["id"]
    print(f"  Apify run: {run_id}")
    elapsed = 0
    while elapsed < APIFY_MAX_WAIT_S:
        time.sleep(APIFY_POLL_S); elapsed += APIFY_POLL_S
        resp = client.get(f"https://api.apify.com/v2/actor-runs/{run_id}", headers=headers)
        status = resp.json()["data"]["status"]
        if status in ("SUCCEEDED", "FAILED", "ABORTED", "TIMED-OUT"):
            break
    if status != "SUCCEEDED":
        print(f"  FAILED: {status}"); client.close(); return []
    cost = resp.json()["data"].get("usageTotalUsd", 0)
    dataset_id = resp.json()["data"]["defaultDatasetId"]
    resp = client.get(f"https://api.apify.com/v2/datasets/{dataset_id}/items?format=json", headers=headers)
    items = resp.json()
    client.close()
    print(f"  Returned {len(items)} items, cost: ${cost:.4f}")
    return items if isinstance(items, list) else []

def fetch_comments(url: str, limit: int = 10) -> list[dict]:
    """Fetch top comments from Apify dataset URL."""
    try:
        resp = httpx.get(url, timeout=15, follow_redirects=True)
        if resp.status_code != 200:
            return []
        comments = resp.json()
        return comments[:limit] if isinstance(comments, list) else []
    except Exception:
        return []

# ── Result saving ───────────────────────────────────────────────────────
def save_result(item_id: str, stage: str, prompt_text: str, api_response: dict,
                parsed: dict, latency_ms: int, upload_ms: int = 0,
                context_used: dict | None = None) -> dict:
    """Save a complete experiment result and return the record."""
    usage = api_response.get("usageMetadata", {})
    in_tok = usage.get("promptTokenCount", 0)
    out_tok = usage.get("candidatesTokenCount", 0)
    record = {
        "item_id": item_id,
        "stage": stage,
        "model": GEMINI_MODEL,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "prompt_text": prompt_text,
        "raw_response": api_response,
        "parsed_output": parsed,
        "metrics": {
            "input_tokens": in_tok, "output_tokens": out_tok,
            "total_tokens": in_tok + out_tok,
            "cost_usd": round(estimate_cost(in_tok, out_tok), 6),
            "latency_ms": latency_ms, "upload_ms": upload_ms,
        },
        "context_used": context_used or {},
    }
    out_dir = RESULTS_DIR / stage
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / f"{item_id}.json", "w") as f:
        json.dump(record, f, indent=2, default=str)
    return record

print(f"Setup complete. Model: {GEMINI_MODEL}")
print(f"Repo root: {REPO_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Export: {EXPORT_PATH} ({'exists' if EXPORT_PATH.exists() else 'MISSING'})")

In [2]:
# Cell 1: Sample Selection — 50 random from recent 500 favorites

with open(EXPORT_PATH) as f:
    export = json.load(f)

favs = export["Likes and Favorites"]["Favorite Videos"]["FavoriteVideoList"]
print(f"Total favorites: {len(favs)}")
print(f"Date range: {favs[-1]['Date']} to {favs[0]['Date']}")

# Take most recent 500 (already sorted newest-first)
recent_500 = favs[:500]
print(f"Recent 500: {recent_500[-1]['Date']} to {recent_500[0]['Date']}")

# Random sample 50 (seeded for reproducibility)
random.seed(42)
sample_50 = random.sample(recent_500, 50)

# Extract platform IDs
selected = []
for item in sample_50:
    url = item.get("Link") or item.get("link", "")
    match = _TIKTOK_VIDEO_ID_RE.search(url)
    pid = match.group(1) if match else None
    selected.append({
        "platform_id": pid,
        "url": url,
        "date": item.get("Date") or item.get("date"),
    })

selected = [s for s in selected if s["platform_id"]]
print(f"\nSelected {len(selected)} items with valid platform IDs")
print(f"Date range: {min(s['date'] for s in selected)} to {max(s['date'] for s in selected)}")

# Preview
for s in selected[:5]:
    print(f"  {s['platform_id']} — {s['date']}")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/s0shaheen/Dev/attic/workbench/workbench/data/my-export/full_anonymized.json'

In [ ]:
# Cell 2: Apify Fetch — get full metadata + media for 50 items

pids = [s["platform_id"] for s in selected]
print(f"Fetching {len(pids)} items from Apify...")
apify_items = apify_fetch(pids)

# Index by platform_id
by_pid = {}
for item in apify_items:
    pid = str(item.get("id", ""))
    if pid:
        by_pid[pid] = item
    web_url = item.get("webVideoUrl", "")
    m = _TIKTOK_VIDEO_ID_RE.search(web_url)
    if m:
        by_pid[m.group(1)] = item

# Build manifest
manifest_items = []
for s in selected:
    pid = s["platform_id"]
    apify = by_pid.get(pid, {})
    is_slideshow = bool(apify.get("isSlideshow"))
    vm = apify.get("videoMeta") or {}
    has_subs = len(vm.get("subtitleLinks") or []) > 0
    caption = apify.get("text", "")

    manifest_items.append({
        "platform_id": pid,
        "matched": bool(apify),
        "date": s["date"],
        "media_type": "slideshow" if is_slideshow else "video",
        "duration": vm.get("duration", 0),
        "has_subtitles": has_subs,
        "text_tier": get_text_tier(caption, has_subs),
        "caption": caption,
        "web_url": apify.get("webVideoUrl", ""),
        "apify_data": apify,
    })

manifest = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "total": len(manifest_items),
    "matched": sum(1 for m in manifest_items if m["matched"]),
    "items": manifest_items,
}
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

# Summary
matched = [m for m in manifest_items if m["matched"]]
videos = [m for m in matched if m["media_type"] == "video"]
slides = [m for m in matched if m["media_type"] == "slideshow"]
tiers = pd.Series([m["text_tier"] for m in matched]).value_counts()

print(f"\nMatched: {len(matched)}/{len(manifest_items)}")
print(f"Videos: {len(videos)}, Slideshows: {len(slides)}")
print(f"With subtitles: {sum(1 for m in matched if m['has_subtitles'])}")
print(f"\nText richness tiers:")
for tier, count in tiers.items():
    print(f"  {tier}: {count}")

In [ ]:
# Cell 3: Download Media + Fetch Comments + Parse Subtitles

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

items = [m for m in manifest["items"] if m["matched"]]
print(f"Downloading media for {len(items)} items...\n")

stats = {"videos": 0, "video_fails": 0, "thumbs": 0, "slides": 0,
         "subs": 0, "comments": 0, "comment_items": 0}

for i, item in enumerate(items):
    pid = item["platform_id"]
    apify = item["apify_data"]
    vm = apify.get("videoMeta") or {}

    # Thumbnail
    cover = vm.get("coverUrl")
    if cover and download_file(cover, MEDIA_DIR / f"thumb_{pid}.jpg"):
        stats["thumbs"] += 1

    # Video
    dl = vm.get("downloadAddr")
    if dl and item["media_type"] == "video":
        if download_file(dl, MEDIA_DIR / f"video_{pid}.mp4"):
            stats["videos"] += 1
        else:
            stats["video_fails"] += 1

    # Slideshow images
    slide_links = apify.get("slideshowImageLinks") or []
    for j, link in enumerate(slide_links):
        url = link.get("downloadLink") or link.get("tiktokLink")
        if url:
            download_file(url, MEDIA_DIR / f"slide_{pid}_{j:03d}.jpg")
    if slide_links:
        stats["slides"] += 1

    # Subtitles — download VTT and parse to text
    sub_links = vm.get("subtitleLinks") or []
    subtitle_text = ""
    for sl in sub_links:
        dl_url = sl.get("downloadLink")
        if dl_url:
            vtt_path = MEDIA_DIR / f"sub_{pid}.vtt"
            if download_file(dl_url, vtt_path):
                subtitle_text = parse_vtt(vtt_path.read_text(errors="replace"))
                stats["subs"] += 1
                break
    item["subtitle_text"] = subtitle_text

    # Comments — fetch from Apify dataset URL
    comments_url = apify.get("commentsDatasetUrl")
    comments = []
    if comments_url:
        comments = fetch_comments(comments_url, limit=10)
        if comments:
            stats["comment_items"] += 1
            stats["comments"] += len(comments)
    item["comments"] = comments

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(items)} processed...")

# Save updated manifest with subtitles + comments
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"\nDownload complete:")
print(f"  Videos: {stats['videos']} OK, {stats['video_fails']} failed")
print(f"  Thumbnails: {stats['thumbs']}")
print(f"  Slideshows: {stats['slides']}")
print(f"  Subtitles: {stats['subs']} items with transcript")
print(f"  Comments: {stats['comments']} total across {stats['comment_items']} items")

In [ ]:
# Cell 4: Stage 1 — Comprehensive Perception

def build_context_block(item: dict) -> str:
    """Build the metadata context block for prompts."""
    apify = item["apify_data"]
    author = apify.get("authorMeta") or {}
    music = apify.get("musicMeta") or {}
    hashtags = [h.get("name", "") for h in (apify.get("hashtags") or []) if isinstance(h, dict)]

    parts = []
    parts.append(f"Caption: {item['caption'][:500]}")
    if hashtags:
        parts.append(f"Hashtags: #{', #'.join(hashtags[:15])}")
    parts.append(f"Creator: @{author.get('name', '?')} ({author.get('nickName', '')})")
    if music.get("musicName"):
        parts.append(f"Music: {music['musicName']} by {music.get('musicAuthor', '?')}")
    parts.append(f"Duration: {item['duration']}s")
    parts.append(f"Engagement: {apify.get('playCount', 0):,} plays, {apify.get('diggCount', 0):,} likes")

    sub = item.get("subtitle_text", "")
    if sub:
        parts.append(f"Subtitle transcript: {sub[:1000]}")

    comments = item.get("comments") or []
    if comments:
        comment_lines = []
        for c in comments[:5]:
            text = c.get("text", "")
            likes = c.get("diggCount", 0)
            comment_lines.append(f'"{text[:100]}" ({likes:,} likes)')
        parts.append("Top comments:\n" + "\n".join(f"  {i+1}. {cl}" for i, cl in enumerate(comment_lines)))

    return "\n".join(parts)


VIDEO_PERCEPTION_PROMPT = """You are analyzing a TikTok video. Produce a detailed perception report — describe everything you observe. Do not classify or judge.

CONTEXT (metadata from the post):
{context}

Watch the video carefully and return JSON:
{{
  "scene_timeline": [
    {{"time_range": "0:00-0:XX", "visual_description": "what is visually happening",
      "audio_description": "what is being said or heard — transcribe key dialogue verbatim",
      "text_on_screen": "transcribe ALL overlaid text, captions, watermarks",
      "key_objects": ["identifiable objects, products, brands visible in this segment"]}}
  ],
  "overall_summary": "comprehensive 3-5 sentence description of the entire video — what is it about, what happens, what is the point?",
  "people": [{{"description": "appearance, clothing, role, actions", "is_creator": true, "speaking": true, "identified_as": "name if recognizable, null otherwise"}}],
  "entities_detected": [
    {{"name": "specific name (not generic descriptions)", "type": "person|place|product|brand|song|book|movie|tv_show|app|restaurant|website",
      "how_identified": "visual|audio|text_overlay|metadata", "confidence": "high|medium|low"}}
  ],
  "presentation_style": {{
    "primary_format": "talking_head|voiceover|text_overlay|cinematic|tutorial_demo|skit|compilation|reaction|slideshow|before_after|pov|other",
    "camera_work": "static|handheld|panning|transitions|split_screen|overhead|selfie",
    "editing_style": "minimal|jump_cuts|heavy_effects|before_after|montage|slow_motion"
  }},
  "audio_profile": {{
    "speech": true,
    "speech_summary": "brief summary of what is said if speech present",
    "music": "none|background|featured|music_video",
    "music_identified": "song name and artist if identifiable",
    "sound_effects": false
  }},
  "visual_mood": "the emotional atmosphere conveyed by visuals, pacing, lighting, and tone",
  "topic_hints": ["2-3 topic areas this content addresses"],
  "affect_hints": ["emotional tones present — how would a viewer FEEL watching this?"],
  "genre_hints": ["content format types — what KIND of TikTok is this?"]
}}

CRITICAL INSTRUCTIONS:
- Be SPECIFIC. Name every identifiable person, place, product, song, show, brand.
- Transcribe key dialogue and speech VERBATIM where possible.
- Transcribe ALL on-screen text including overlays, captions, watermarks.
- If you recognize a celebrity, character, TV show, movie, or song — NAME IT with confidence level.
- Distinguish between what you SEE, what you HEAR, and what you INFER from metadata.
- Precision over hedging. "Tony Soprano from The Sopranos" not "a man who appears to be a character"."""


def build_slideshow_prompt(n_images: int) -> str:
    """Build slideshow perception prompt with correct image count."""
    image_instruction = "each image carefully, one by one" if n_images > 1 else "the image"
    media_desc = f"photo carousel with {n_images} images" if n_images > 1 else "image post"

    return f"""You are analyzing a TikTok {media_desc}. Produce a detailed perception report — describe everything you observe. Do not classify or judge.

CONTEXT (metadata from the post):
{{context}}

Analyze {image_instruction} and return JSON:
{{{{
  "per_image": [
    {{{{"image_number": 1, "description": "detailed description of what the image shows",
      "text_detected": "transcribe ALL readable text exactly as written",
      "objects": ["every identifiable object, product, brand, logo"],
      "entities": ["named things: people, places, products, restaurants, apps"]}}}}
  ],
  "overall_summary": "comprehensive 3-5 sentence description — what is this post about, what is the point?",
  "narrative_thread": "what story, message, or recommendation do these images convey together?",
  "entities_detected": [
    {{{{"name": "specific name", "type": "person|place|product|brand|song|book|movie|tv_show|app|restaurant|website",
      "how_identified": "visual|text_overlay|metadata", "confidence": "high|medium|low"}}}}
  ],
  "presentation_style": {{{{
    "primary_format": "text_cards|photo_dump|product_showcase|infographic|meme|screenshot|recommendation_list|room_tour|outfit_layout|other",
    "visual_style": "minimal|aesthetic|informational|meme|collage|professional|casual"
  }}}},
  "visual_mood": "the emotional atmosphere and aesthetic of the images",
  "topic_hints": ["2-3 topic areas"],
  "affect_hints": ["emotional tones — how would a viewer FEEL looking at this?"],
  "genre_hints": ["content format types — what KIND of TikTok is this?"]
}}}}

CRITICAL INSTRUCTIONS:
- Transcribe ALL visible text in every image — menus, signs, labels, captions, watermarks, everything.
- Name every identifiable person, place, product, restaurant, brand, logo.
- For recommendation/list posts: extract every specific item recommended (restaurant names, product names, addresses).
- For outfit/room posts: describe specific items, brands, colors, styles.
- Precision over hedging."""

# Load manifest
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

items = [m for m in manifest["items"] if m["matched"]]
print(f"Running Stage 1 perception on {len(items)} items...\n")

perception_results = []
total_cost = 0.0

for i, item in enumerate(items):
    pid = item["platform_id"]
    context = build_context_block(item)
    has_subs = bool(item.get("subtitle_text"))
    has_comments = bool(item.get("comments"))
    ctx_info = {"has_caption": bool(item["caption"]), "has_subtitles": has_subs,
                "has_comments": has_comments,
                "subtitle_length": len(item.get("subtitle_text", "")),
                "comment_count": len(item.get("comments", [])),
                "text_richness_tier": item["text_tier"]}

    start = time.time()
    upload_ms = 0

    try:
        if item["media_type"] == "video":
            video_path = MEDIA_DIR / f"video_{pid}.mp4"
            if not video_path.exists():
                print(f"  [{i+1}] {pid}: SKIP (no video file)")
                continue

            prompt = VIDEO_PERCEPTION_PROMPT.format(context=context)
            up_start = time.time()
            file_name = gemini_upload_file(str(video_path), "video/mp4")
            file_uri = gemini_wait_for_file(file_name)
            upload_ms = int((time.time() - up_start) * 1000)

            parts = [{"text": prompt}, {"fileData": {"mimeType": "video/mp4", "fileUri": file_uri}}]
            api_resp = gemini_generate(parts)
            gemini_delete_file(file_name)

        else:  # slideshow / image
            slide_images = sorted(MEDIA_DIR.glob(f"slide_{pid}_*.jpg"))
            if not slide_images:
                thumb = MEDIA_DIR / f"thumb_{pid}.jpg"
                if not thumb.exists():
                    print(f"  [{i+1}] {pid}: SKIP (no images)")
                    continue
                slide_images = [thumb]

            n_images = len(slide_images)
            prompt = build_slideshow_prompt(n_images).replace("{context}", context)

            parts = [{"text": prompt}]
            for img_path in slide_images:
                parts.append(image_to_inline(img_path))
            api_resp = gemini_generate(parts)

        latency_ms = int((time.time() - start) * 1000)
        text = api_resp["candidates"][0]["content"]["parts"][0]["text"]
        parsed = safe_json_parse(text)

        record = save_result(pid, "perception", prompt, api_resp, parsed, latency_ms, upload_ms, ctx_info)
        perception_results.append(record)
        cost = record["metrics"]["cost_usd"]
        total_cost += cost

        n_ent = len(parsed.get("entities_detected", []))
        has_err = "_parse_error" in parsed
        status = f"{'PARSE_ERR' if has_err else 'OK'} | {record['metrics']['input_tokens']}+{record['metrics']['output_tokens']} tok | {n_ent} entities | ${cost:.4f}"
        print(f"  [{i+1}/{len(items)}] {pid}: {status}")

    except Exception as e:
        latency_ms = int((time.time() - start) * 1000)
        print(f"  [{i+1}/{len(items)}] {pid}: ERROR — {str(e)[:80]}")
        save_result(pid, "perception", "", {}, {"_error": str(e)}, latency_ms, upload_ms, ctx_info)

    time.sleep(0.5)

parse_errors = sum(1 for r in perception_results if "_parse_error" in r.get("parsed_output", {}))
print(f"\nStage 1 complete: {len(perception_results)} items, ${total_cost:.4f} total, {parse_errors} parse errors")

In [ ]:
# Cell 5: Stage 2 — 5-Facet Classification (vision-informed)

from app.services.ontology import ONTOLOGY_V1, FACET_NAMES, format_ontology_for_prompt

FIVE_FACETS = ["topic", "genre", "affect", "presentation_style", "content_provenance"]

def build_5facet_ontology() -> str:
    lines = ["## Classification Ontology (5 Facets)\n"]
    for facet in FIVE_FACETS:
        labels = ONTOLOGY_V1[facet]
        lines.append(f"**{facet}** — pick exactly ONE:\n  {', '.join(labels)}\n")
    lines.append("""INSTRUCTIONS:
- For each facet, select the single best tier-1 label from the list above.
- You may also suggest 1-3 "micro_labels" — free-form refinements that add nuance (e.g., topic="food" with micro_labels=["pasta recipe", "Italian cooking"]).
- Confidence: 0.0-1.0 where 0.5 = uncertain/ambiguous, 0.8 = confident, 0.95 = very sure.
- If the content genuinely fits multiple labels, pick the dominant one and use micro_labels for the secondary.
- If no label fits well, use "other" and explain in micro_labels.

Return JSON with the 5 facet names as keys:
{
  "topic": {"label": "food", "micro_labels": ["pasta recipe"], "confidence": 0.9},
  "genre": {"label": "recipe", "micro_labels": ["step-by-step cooking"], "confidence": 0.85},
  ...
}""")
    return "\n".join(lines)

ONTOLOGY_5 = build_5facet_ontology()

CLASSIFY_VISION_PROMPT = """Classify this TikTok content using the following ontology.

{ontology}

ALL AVAILABLE CONTEXT:
- Vision analysis (Stage 1 perception): {perception}
- Caption: {caption}
- Hashtags: {hashtags}
- Creator: @{username}
- Music: {music}
- Subtitle transcript: {subtitles}
- Top comments: {comments}
- Engagement: {play_count} plays, {like_count} likes

Use ALL the context above — especially the vision analysis — to make your classification decisions.
Return ONLY valid JSON."""

# Load perception results
perception_by_pid = {}
for f in sorted((RESULTS_DIR / "perception").glob("*.json")):
    with open(f) as fh:
        r = json.load(fh)
    perception_by_pid[r["item_id"]] = r

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

items = [m for m in manifest["items"] if m["matched"] and m["platform_id"] in perception_by_pid]
print(f"Running 5-facet classification (vision-informed) on {len(items)} items...\n")

classify_results = []
total_cost = 0.0

for i, item in enumerate(items):
    pid = item["platform_id"]
    perc = perception_by_pid[pid]
    perc_output = perc.get("parsed_output", {})
    perc_text = json.dumps(perc_output, indent=1, default=str)[:4000]

    apify = item["apify_data"]
    author = apify.get("authorMeta") or {}
    music = apify.get("musicMeta") or {}
    hashtags = [h.get("name", "") for h in (apify.get("hashtags") or []) if isinstance(h, dict)]
    comments = item.get("comments") or []
    comment_text = "; ".join(f'"{c.get("text", "")[:80]}"' for c in comments[:5]) if comments else "(none)"

    prompt = CLASSIFY_VISION_PROMPT.format(
        ontology=ONTOLOGY_5, perception=perc_text, caption=item["caption"][:300],
        hashtags=", ".join(hashtags[:10]) if hashtags else "(none)",
        username=author.get("name", "?"), music=music.get("musicName", "(none)"),
        subtitles=item.get("subtitle_text", "(none)")[:500],
        comments=comment_text,
        play_count=f"{apify.get('playCount', 0):,}",
        like_count=f"{apify.get('diggCount', 0):,}",
    )

    start = time.time()
    try:
        api_resp = gemini_generate([{"text": prompt}], max_tokens=1024)
        latency_ms = int((time.time() - start) * 1000)
        text = api_resp["candidates"][0]["content"]["parts"][0]["text"]
        parsed = safe_json_parse(text)

        record = save_result(pid, "classify_5facet_vision", prompt, api_resp, parsed, latency_ms,
                             context_used={"has_perception": True, "text_richness_tier": item["text_tier"]})
        classify_results.append(record)
        cost = record["metrics"]["cost_usd"]
        total_cost += cost

        labels = {f: parsed.get(f, {}).get("label", "?") for f in FIVE_FACETS}
        print(f"  [{i+1}/{len(items)}] {pid}: {labels} | ${cost:.4f}")

    except Exception as e:
        latency_ms = int((time.time() - start) * 1000)
        print(f"  [{i+1}/{len(items)}] {pid}: ERROR — {str(e)[:80]}")

    time.sleep(0.3)

print(f"\n5-facet vision classification complete: {len(classify_results)} items, ${total_cost:.4f}")

In [ ]:
# Cell 6: Stage 2 — 5-Facet Classification (text-only baseline, NO vision)

CLASSIFY_TEXT_PROMPT = """Classify this TikTok content using the following ontology.
You do NOT have visual analysis available — classify based on text signals only.

{ontology}

ALL AVAILABLE TEXT CONTEXT:
- Caption: {caption}
- Hashtags: {hashtags}
- Creator: @{username}
- Music: {music}
- Subtitle transcript: {subtitles}
- Top comments: {comments}
- Engagement: {play_count} plays, {like_count} likes

Use ALL the text context above to make your best classification decisions.
Where text signals are ambiguous, set lower confidence scores.
Return ONLY valid JSON."""

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

items = [m for m in manifest["items"] if m["matched"]]
print(f"Running 5-facet classification (text-only) on {len(items)} items...\n")

text_classify_results = []
total_cost = 0.0

for i, item in enumerate(items):
    pid = item["platform_id"]
    apify = item["apify_data"]
    author = apify.get("authorMeta") or {}
    music = apify.get("musicMeta") or {}
    hashtags = [h.get("name", "") for h in (apify.get("hashtags") or []) if isinstance(h, dict)]
    comments = item.get("comments") or []
    comment_text = "; ".join(f'"{c.get("text", "")[:80]}"' for c in comments[:5]) if comments else "(none)"

    prompt = CLASSIFY_TEXT_PROMPT.format(
        ontology=ONTOLOGY_5, caption=item["caption"][:300],
        hashtags=", ".join(hashtags[:10]) if hashtags else "(none)",
        username=author.get("name", "?"), music=music.get("musicName", "(none)"),
        subtitles=item.get("subtitle_text", "(none)")[:500],
        comments=comment_text,
        play_count=f"{apify.get('playCount', 0):,}",
        like_count=f"{apify.get('diggCount', 0):,}",
    )

    start = time.time()
    try:
        api_resp = gemini_generate([{"text": prompt}], max_tokens=1024)
        latency_ms = int((time.time() - start) * 1000)
        text = api_resp["candidates"][0]["content"]["parts"][0]["text"]
        parsed = safe_json_parse(text)

        record = save_result(pid, "classify_5facet_text", prompt, api_resp, parsed, latency_ms,
                             context_used={"has_perception": False, "text_richness_tier": item["text_tier"]})
        text_classify_results.append(record)
        cost = record["metrics"]["cost_usd"]
        total_cost += cost

        labels = {f: parsed.get(f, {}).get("label", "?") for f in FIVE_FACETS}
        print(f"  [{i+1}/{len(items)}] {pid}: {labels} | ${cost:.4f}")

    except Exception as e:
        print(f"  [{i+1}/{len(items)}] {pid}: ERROR — {str(e)[:80]}")

    time.sleep(0.3)

print(f"\nText-only classification complete: {len(text_classify_results)} items, ${total_cost:.4f}")

In [ ]:
# Cell 7: 8-Facet Classification + Inter-Facet Correlation (redundancy test)

from sklearn.metrics import normalized_mutual_info_score
from sklearn.preprocessing import LabelEncoder

ALL_FACETS = list(FACET_NAMES) if hasattr(FACET_NAMES, '__iter__') else list(ONTOLOGY_V1.keys())

def build_8facet_ontology() -> str:
    lines = ["## Classification Ontology (8 Facets)\n"]
    for facet in ALL_FACETS:
        labels = ONTOLOGY_V1[facet]
        lines.append(f"**{facet}**: {', '.join(labels)}")
    lines.append("\nFor each facet, pick exactly one label. Also suggest micro-labels and confidence.")
    lines.append('Return JSON with facet names as keys, each containing: "label", "micro_labels", "confidence".')
    return "\n".join(lines)

ONTOLOGY_8 = build_8facet_ontology()

# Reuse perception results from Cell 4
items_with_perc = [m for m in manifest["items"] if m["matched"] and m["platform_id"] in perception_by_pid]
print(f"Running 8-facet classification on {len(items_with_perc)} items...\n")

eight_facet_results = []
total_cost = 0.0

for i, item in enumerate(items_with_perc):
    pid = item["platform_id"]
    perc = perception_by_pid[pid]
    perc_text = json.dumps(perc.get("parsed_output", {}), indent=1, default=str)[:3000]

    apify = item["apify_data"]
    author = apify.get("authorMeta") or {}
    music = apify.get("musicMeta") or {}
    hashtags = [h.get("name", "") for h in (apify.get("hashtags") or []) if isinstance(h, dict)]
    comments = item.get("comments") or []
    comment_text = "; ".join(f'"{c.get("text", "")[:80]}"' for c in comments[:5]) if comments else "(none)"

    prompt = CLASSIFY_VISION_PROMPT.format(
        ontology=ONTOLOGY_8, perception=perc_text, caption=item["caption"][:300],
        hashtags=", ".join(hashtags[:10]) if hashtags else "(none)",
        username=author.get("name", "?"), music=music.get("musicName", "(none)"),
        subtitles=item.get("subtitle_text", "(none)")[:500],
        comments=comment_text,
        play_count=f"{apify.get('playCount', 0):,}",
        like_count=f"{apify.get('diggCount', 0):,}",
    )

    start = time.time()
    try:
        api_resp = gemini_generate([{"text": prompt}], max_tokens=1536)
        latency_ms = int((time.time() - start) * 1000)
        text = api_resp["candidates"][0]["content"]["parts"][0]["text"]
        parsed = safe_json_parse(text)

        record = save_result(pid, "classify_8facet_vision", prompt, api_resp, parsed, latency_ms,
                             context_used={"has_perception": True, "text_richness_tier": item["text_tier"]})
        eight_facet_results.append(record)
        total_cost += record["metrics"]["cost_usd"]

    except Exception as e:
        print(f"  [{i+1}] {pid}: ERROR — {str(e)[:60]}")
    time.sleep(0.3)

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(items_with_perc)} done...")

print(f"\n8-facet classification complete: {len(eight_facet_results)} items, ${total_cost:.4f}")

# ── Inter-facet correlation ─────────────────────────────────────────────
print("\n=== Inter-Facet Correlation (NMI) ===")
print("NMI > 0.5 suggests redundancy\n")

# Collect labels
facet_labels = {f: [] for f in ALL_FACETS}
valid_items = []
for r in eight_facet_results:
    parsed = r.get("parsed_output", {})
    if "_parse_error" in parsed:
        continue
    valid = True
    for f in ALL_FACETS:
        label = parsed.get(f, {}).get("label") if isinstance(parsed.get(f), dict) else None
        if not label:
            valid = False
            break
        facet_labels[f].append(label)
    if valid:
        valid_items.append(r)

if len(valid_items) >= 10:
    pairs_to_test = [
        ("genre", "communicative_intent"),
        ("topic", "communicative_intent"),
        ("genre", "creator_role"),
        ("affect", "viewer_orientation"),
        ("presentation_style", "genre"),
        ("topic", "genre"),
    ]
    for fa, fb in pairs_to_test:
        if fa in facet_labels and fb in facet_labels and len(facet_labels[fa]) == len(facet_labels[fb]):
            le_a = LabelEncoder().fit_transform(facet_labels[fa])
            le_b = LabelEncoder().fit_transform(facet_labels[fb])
            nmi = normalized_mutual_info_score(le_a, le_b)
            flag = " ⚠️ REDUNDANT" if nmi > 0.5 else ""
            print(f"  {fa} ↔ {fb}: NMI = {nmi:.3f}{flag}")
else:
    print(f"  Only {len(valid_items)} valid items — need ≥10 for meaningful NMI")

In [ ]:
# Cell 8: Slideshow Image Informativeness Assessment

INFORMATIVENESS_PROMPT = """For each image in this TikTok carousel, classify it as:
- "informational" — contains unique content, text, recommendations, data, products, or substantive visual information
- "decorative" — title card, "follow me" card, aesthetic filler, emoji-only, duplicate of another image, or no unique content

Return JSON:
{{"images": [{{"index": 1, "type": "informational|decorative", "reason": "brief reason"}}]}}"""

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

slideshow_items = [m for m in manifest["items"] if m["matched"] and m["media_type"] == "slideshow"]
print(f"Assessing image informativeness for {len(slideshow_items)} slideshows...\n")

info_results = []
total_informational = 0
total_decorative = 0
total_images = 0

for i, item in enumerate(slideshow_items):
    pid = item["platform_id"]
    slide_images = sorted(MEDIA_DIR.glob(f"slide_{pid}_*.jpg"))

    if len(slide_images) < 2:
        print(f"  [{i+1}] {pid}: SKIP (only {len(slide_images)} images)")
        continue

    parts = [{"text": INFORMATIVENESS_PROMPT}]
    for img_path in slide_images:
        parts.append(image_to_inline(img_path))

    start = time.time()
    try:
        api_resp = gemini_generate(parts, max_tokens=1024)
        latency_ms = int((time.time() - start) * 1000)
        text = api_resp["candidates"][0]["content"]["parts"][0]["text"]
        parsed = safe_json_parse(text)

        record = save_result(pid, "slideshow_informativeness", INFORMATIVENESS_PROMPT, api_resp, parsed, latency_ms)
        info_results.append(record)

        images = parsed.get("images", [])
        n_info = sum(1 for img in images if img.get("type") == "informational")
        n_deco = sum(1 for img in images if img.get("type") == "decorative")
        total_informational += n_info
        total_decorative += n_deco
        total_images += len(slide_images)

        print(f"  [{i+1}] {pid}: {len(slide_images)} images → {n_info} informational, {n_deco} decorative")

    except Exception as e:
        print(f"  [{i+1}] {pid}: ERROR — {str(e)[:60]}")

    time.sleep(0.3)

print(f"\nSlideshow informativeness complete:")
print(f"  Total images assessed: {total_images}")
print(f"  Informational: {total_informational} ({total_informational/total_images*100:.0f}%)" if total_images else "")
print(f"  Decorative: {total_decorative} ({total_decorative/total_images*100:.0f}%)" if total_images else "")
if total_decorative > 0:
    print(f"  → {total_decorative/total_images*100:.0f}% of carousel images could be skipped in production")

In [ ]:
# Cell 9: Results Summary + Stratified Analysis

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Load all results
def load_results(stage: str) -> dict[str, dict]:
    results = {}
    stage_dir = RESULTS_DIR / stage
    if not stage_dir.exists():
        return results
    for f in stage_dir.glob("*.json"):
        with open(f) as fh:
            r = json.load(fh)
        results[r["item_id"]] = r
    return results

perc = load_results("perception")
cls_vision = load_results("classify_5facet_vision")
cls_text = load_results("classify_5facet_text")
cls_8f = load_results("classify_8facet_vision")

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
items = [m for m in manifest["items"] if m["matched"]]

# ── Overall stats ───────────────────────────────────────────────────────
total_cost = sum(r["metrics"]["cost_usd"] for r in perc.values())
total_cost += sum(r["metrics"]["cost_usd"] for r in cls_vision.values())
total_cost += sum(r["metrics"]["cost_usd"] for r in cls_text.values())
total_cost += sum(r["metrics"]["cost_usd"] for r in cls_8f.values())

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"  Items processed: {len(perc)}")
print(f"  Total cost: ${total_cost:.4f}")
print(f"  Perception results: {len(perc)}")
print(f"  5-facet vision classification: {len(cls_vision)}")
print(f"  5-facet text-only classification: {len(cls_text)}")
print(f"  8-facet classification: {len(cls_8f)}")

# ── Stratified analysis by text-richness tier ───────────────────────────
print(f"\n{'='*60}")
print("STRATIFIED ANALYSIS BY TEXT-RICHNESS TIER")
print("=" * 60)

tiers = {"rich_text": [], "caption_only": [], "low_text": []}
for item in items:
    pid = item["platform_id"]
    tier = item["text_tier"]
    if pid in perc and pid in cls_vision and pid in cls_text:
        tiers[tier].append(pid)

for tier_name, pids in tiers.items():
    if not pids:
        continue
    print(f"\n--- {tier_name} ({len(pids)} items) ---")

    # Entity counts from perception
    ent_counts = [len(perc[pid].get("parsed_output", {}).get("entities_detected", [])) for pid in pids]
    print(f"  Avg entities (Stage 1): {np.mean(ent_counts):.1f}")

    # Classification agreement: how often do vision and text-only produce the same label?
    agreements = {f: 0 for f in FIVE_FACETS}
    total_compared = 0
    for pid in pids:
        v = cls_vision.get(pid, {}).get("parsed_output", {})
        t = cls_text.get(pid, {}).get("parsed_output", {})
        if "_parse_error" in v or "_parse_error" in t:
            continue
        total_compared += 1
        for f in FIVE_FACETS:
            v_label = v.get(f, {}).get("label") if isinstance(v.get(f), dict) else None
            t_label = t.get(f, {}).get("label") if isinstance(t.get(f), dict) else None
            if v_label and t_label and v_label == t_label:
                agreements[f] += 1

    if total_compared > 0:
        print(f"  Classification agreement (vision vs text-only):")
        for f in FIVE_FACETS:
            pct = agreements[f] / total_compared * 100
            print(f"    {f}: {agreements[f]}/{total_compared} ({pct:.0f}%)")

    # Avg perception cost
    costs = [perc[pid]["metrics"]["cost_usd"] for pid in pids]
    print(f"  Avg perception cost: ${np.mean(costs):.4f}")

# ── Vision vs Text-Only disagreement analysis ───────────────────────────
print(f"\n{'='*60}")
print("VISION vs TEXT-ONLY: WHERE DO THEY DISAGREE?")
print("=" * 60)

for f in FIVE_FACETS:
    disagreements = []
    for item in items:
        pid = item["platform_id"]
        if pid not in cls_vision or pid not in cls_text:
            continue
        v = cls_vision[pid].get("parsed_output", {})
        t = cls_text[pid].get("parsed_output", {})
        v_label = v.get(f, {}).get("label") if isinstance(v.get(f), dict) else None
        t_label = t.get(f, {}).get("label") if isinstance(t.get(f), dict) else None
        if v_label and t_label and v_label != t_label:
            disagreements.append((pid, v_label, t_label, item["text_tier"]))

    if disagreements:
        print(f"\n  {f}: {len(disagreements)} disagreements")
        for pid, v_lab, t_lab, tier in disagreements[:5]:
            print(f"    {pid} [{tier}]: vision={v_lab}, text={t_lab}")

print("\nResults saved to:", RESULTS_DIR)

In [ ]:
# Cell 10: Generate Eval UI (self-contained HTML)

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

perc = load_results("perception")
cls_v = load_results("classify_5facet_vision")
cls_t = load_results("classify_5facet_text")

# Build eval data for the UI
eval_items = []
for item in manifest["items"]:
    if not item["matched"]:
        continue
    pid = item["platform_id"]
    if pid not in perc:
        continue

    p = perc[pid].get("parsed_output", {})
    cv = cls_v.get(pid, {}).get("parsed_output", {})
    ct = cls_t.get(pid, {}).get("parsed_output", {})

    eval_items.append({
        "id": pid,
        "web_url": item.get("web_url", ""),
        "caption": item["caption"][:300],
        "hashtags": [h.get("name","") for h in (item["apify_data"].get("hashtags") or []) if isinstance(h, dict)][:10],
        "media_type": item["media_type"],
        "duration": item["duration"],
        "text_tier": item["text_tier"],
        "has_subtitles": item.get("has_subtitles", False),
        "perception_summary": p.get("overall_summary", p.get("_raw_text", "")[:500] if "_raw_text" in p else json.dumps(p)[:500]),
        "perception_entities": p.get("entities_detected", []),
        "perception_scenes": p.get("scene_timeline", []),
        "perception_style": p.get("presentation_style", {}),
        "perception_mood": p.get("visual_mood", ""),
        "perception_topics": p.get("topic_hints", []),
        "perception_affects": p.get("affect_hints", []),
        "perception_genres": p.get("genre_hints", []),
        "classify_vision": {f: cv.get(f, {}) for f in FIVE_FACETS} if cv and "_parse_error" not in cv else {},
        "classify_text": {f: ct.get(f, {}) for f in FIVE_FACETS} if ct and "_parse_error" not in ct else {},
        "metrics": perc[pid].get("metrics", {}),
    })

eval_json = json.dumps(eval_items, indent=None, default=str)

html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Attic Vision Eval — Day 2 v2</title>
<style>
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: 'DM Sans', -apple-system, sans-serif; background: #F8F7F4; color: #1C1B18; padding: 20px; }
.header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px; padding: 16px; background: white; border: 0.5px solid #E6E4DE; border-radius: 8px; }
.header h1 { font-size: 18px; font-weight: 500; }
.progress { font-size: 14px; color: #9C9890; }
.nav { display: flex; gap: 8px; }
.nav button { padding: 8px 16px; border: 0.5px solid #E6E4DE; border-radius: 6px; background: white; cursor: pointer; font-size: 14px; }
.nav button:hover { background: #F0EEE8; }
.card { background: white; border: 0.5px solid #E6E4DE; border-radius: 8px; padding: 24px; margin-bottom: 16px; }
.meta { display: flex; gap: 12px; flex-wrap: wrap; margin: 8px 0; }
.tag { padding: 2px 8px; border-radius: 4px; font-size: 12px; background: #F0EEE8; color: #2C2926; }
.tag.tier-rich_text { background: #d4edda; }
.tag.tier-caption_only { background: #fff3cd; }
.tag.tier-low_text { background: #f8d7da; }
.section { margin-top: 16px; }
.section h3 { font-size: 14px; color: #9C9890; margin-bottom: 8px; text-transform: uppercase; letter-spacing: 0.5px; }
.perception-text { font-size: 14px; line-height: 1.6; white-space: pre-wrap; background: #FAFAF8; padding: 12px; border-radius: 6px; max-height: 300px; overflow-y: auto; }
.entities { display: flex; flex-wrap: wrap; gap: 6px; margin-top: 8px; }
.entity { padding: 3px 8px; border-radius: 4px; font-size: 12px; background: #E6E4DE; }
.classify-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; margin-top: 8px; }
.classify-col h4 { font-size: 13px; margin-bottom: 8px; color: #9C9890; }
.facet-row { display: flex; justify-content: space-between; padding: 4px 0; font-size: 13px; border-bottom: 1px solid #F0EEE8; }
.facet-label { font-weight: 500; }
.facet-value { color: #9C9890; }
.rubric { margin-top: 20px; padding: 16px; background: #FAFAF8; border-radius: 6px; }
.rubric h3 { margin-bottom: 12px; }
.dim-row { display: flex; align-items: center; gap: 12px; margin-bottom: 10px; }
.dim-name { width: 160px; font-size: 13px; font-weight: 500; }
.dim-buttons { display: flex; gap: 4px; }
.dim-buttons button { width: 36px; height: 36px; border: 1.5px solid #E6E4DE; border-radius: 6px; background: white; cursor: pointer; font-size: 14px; font-weight: 500; transition: all 0.15s; }
.dim-buttons button:hover { background: #F0EEE8; }
.dim-buttons button.selected { background: #2C2926; color: white; border-color: #2C2926; }
.dim-hint { font-size: 11px; color: #9C9890; flex: 1; }
.notes-area { width: 100%; margin-top: 12px; padding: 8px; border: 0.5px solid #E6E4DE; border-radius: 6px; font-size: 13px; min-height: 60px; resize: vertical; font-family: inherit; }
.total-score { font-size: 16px; font-weight: 500; margin-top: 12px; }
.export-btn { padding: 10px 20px; background: #2C2926; color: white; border: none; border-radius: 6px; cursor: pointer; font-size: 14px; margin-top: 12px; }
.export-btn:hover { background: #1C1B18; }
.tiktok-embed { margin: 12px 0; }
.tiktok-embed iframe { width: 325px; height: 580px; border: none; border-radius: 8px; }
</style>
</head>
<body>

<div class="header">
  <h1>Attic Vision Eval — Day 2 v2</h1>
  <span class="progress" id="progress">0/0 scored</span>
  <div class="nav">
    <button onclick="prev()">← Prev</button>
    <button onclick="next()">Next →</button>
    <button class="export-btn" onclick="exportScores()">Export Scores</button>
  </div>
</div>

<div id="content"></div>

<script>
const ITEMS = """ + eval_json + """;

const DIMS = [
  {key: "accuracy", name: "Accuracy", hints: ["1: Major errors", "2: Mostly correct", "3: Everything checks out"]},
  {key: "completeness", name: "Completeness", hints: ["1: Surface only", "2: Got gist, missed secondary", "3: Full content understood"]},
  {key: "specificity", name: "Specificity", hints: ["1: All vague", "2: Mix named+generic", "3: Named everything"]},
  {key: "entity_coverage", name: "Entity Coverage", hints: ["1: Missed obvious", "2: Got major ones", "3: Caught everything"]},
  {key: "classification_signal", name: "Class. Signal", hints: ["1: Topic ambiguous", "2: Topic/Genre clear", "3: All 5 facets classifiable"]},
  {key: "vision_added_value", name: "Vision Added Value", hints: ["1: Nothing beyond text", "2: Useful detail added", "3: Major invisible content revealed"]},
];

let currentIdx = 0;
let scores = JSON.parse(localStorage.getItem("attic_eval_v2") || "{}");

function render() {
  const item = ITEMS[currentIdx];
  const s = scores[item.id] || {};
  const entities = (item.perception_entities || []).slice(0, 15);
  const scenes = (item.perception_scenes || []).slice(0, 8);

  let scenesHtml = scenes.map(sc => {
    const ts = sc.time_range || sc.timestamp || "";
    const vis = sc.visual_description || sc.description || "";
    const aud = sc.audio_description || "";
    return `<div style="margin:4px 0;font-size:13px"><b>${ts}</b>: ${vis}${aud ? '<br><i>Audio: '+aud+'</i>' : ''}</div>`;
  }).join("");

  let classifyHtml = "";
  const facets = ["topic","genre","affect","presentation_style","content_provenance"];
  if (Object.keys(item.classify_vision).length || Object.keys(item.classify_text).length) {
    classifyHtml = `<div class="classify-grid">
      <div class="classify-col"><h4>Vision-Informed</h4>${facets.map(f => {
        const v = item.classify_vision[f] || {};
        return `<div class="facet-row"><span class="facet-label">${f}</span><span class="facet-value">${v.label||'?'} (${(v.confidence||0).toFixed(2)})</span></div>`;
      }).join("")}</div>
      <div class="classify-col"><h4>Text-Only</h4>${facets.map(f => {
        const v = item.classify_text[f] || {};
        return `<div class="facet-row"><span class="facet-label">${f}</span><span class="facet-value">${v.label||'?'} (${(v.confidence||0).toFixed(2)})</span></div>`;
      }).join("")}</div>
    </div>`;
  }

  let rubricHtml = DIMS.map(d => {
    const val = s[d.key] || 0;
    return `<div class="dim-row">
      <span class="dim-name">${d.name}</span>
      <div class="dim-buttons">
        ${[1,2,3].map(n => `<button class="${val===n?'selected':''}" onclick="setScore('${d.key}',${n})">${n}</button>`).join("")}
      </div>
      <span class="dim-hint">${d.hints[val-1]||''}</span>
    </div>`;
  }).join("");

  const total = DIMS.reduce((sum,d) => sum + (s[d.key]||0), 0);
  const maxTotal = DIMS.length * 3;

  document.getElementById("content").innerHTML = `
    <div class="card">
      <h2>${currentIdx+1}/${ITEMS.length} — ${item.id}</h2>
      <div class="meta">
        <span class="tag">${item.media_type}</span>
        <span class="tag">${item.duration}s</span>
        <span class="tag tier-${item.text_tier}">${item.text_tier}</span>
        ${item.has_subtitles ? '<span class="tag">has subs</span>' : ''}
      </div>
      <p style="margin:8px 0;font-size:14px"><b>Caption:</b> ${item.caption}</p>
      <p style="font-size:13px;color:#9C9890"><b>Hashtags:</b> ${item.hashtags.map(h=>'#'+h).join(' ')}</p>
      ${item.web_url ? `<div class="tiktok-embed"><a href="${item.web_url}" target="_blank" style="color:#A06840">Open on TikTok ↗</a></div>` : ''}
    </div>

    <div class="card">
      <div class="section">
        <h3>Stage 1: Perception (${item.media_type === 'video' ? 'full video' : 'all images'})</h3>
        <div class="perception-text">${item.perception_summary || '(no summary)'}</div>
        ${scenesHtml ? `<div style="margin-top:12px"><b>Scenes:</b>${scenesHtml}</div>` : ''}
        ${entities.length ? `<div class="entities">${entities.map(e => `<span class="entity">${e.name||'?'} (${e.type||'?'})</span>`).join("")}</div>` : ''}
        <p style="margin-top:8px;font-size:13px;color:#9C9890">
          <b>Style:</b> ${JSON.stringify(item.perception_style)} |
          <b>Mood:</b> ${item.perception_mood} |
          <b>Topics:</b> ${(item.perception_topics||[]).join(', ')} |
          <b>Cost:</b> $${(item.metrics.cost_usd||0).toFixed(4)}
        </p>
      </div>

      <div class="section">
        <h3>Stage 2: Classification (vision vs text-only)</h3>
        ${classifyHtml || '<p style="color:#9C9890">No classification results</p>'}
      </div>
    </div>

    <div class="card rubric">
      <h3>Score This Item (1=Poor, 2=Adequate, 3=Strong)</h3>
      ${rubricHtml}
      <textarea class="notes-area" placeholder="Notes (optional)..." oninput="setNotes(this.value)">${s.notes||''}</textarea>
      <div class="total-score">Total: ${total}/${maxTotal}</div>
    </div>`;

  updateProgress();
}

function setScore(dim, val) {
  const item = ITEMS[currentIdx];
  if (!scores[item.id]) scores[item.id] = {};
  scores[item.id][dim] = val;
  localStorage.setItem("attic_eval_v2", JSON.stringify(scores));
  render();
}

function setNotes(text) {
  const item = ITEMS[currentIdx];
  if (!scores[item.id]) scores[item.id] = {};
  scores[item.id].notes = text;
  localStorage.setItem("attic_eval_v2", JSON.stringify(scores));
}

function prev() { if (currentIdx > 0) { currentIdx--; render(); window.scrollTo(0,0); } }
function next() { if (currentIdx < ITEMS.length-1) { currentIdx++; render(); window.scrollTo(0,0); } }

function updateProgress() {
  const scored = Object.values(scores).filter(s => DIMS.some(d => s[d.key])).length;
  document.getElementById("progress").textContent = `${scored}/${ITEMS.length} scored`;
}

function exportScores() {
  const blob = new Blob([JSON.stringify(scores, null, 2)], {type: "application/json"});
  const url = URL.createObjectURL(blob);
  const a = document.createElement("a");
  a.href = url; a.download = "eval_scores_v2.json"; a.click();
  URL.revokeObjectURL(url);
}

document.addEventListener("keydown", e => {
  if (e.key === "ArrowLeft") prev();
  if (e.key === "ArrowRight") next();
});

render();
</script>
</body>
</html>"""

eval_path = DATA_DIR / "eval_ui.html"
eval_path.write_text(html)
print(f"Eval UI saved to: {eval_path}")
print(f"Items included: {len(eval_items)}")
print(f"Open in browser: file://{eval_path}")
print(f"\nScores auto-save to localStorage. Use arrow keys to navigate. Click Export to download JSON.")